# Lab 06 solution, notebook 2: give the desk a fourth lookup

The worked answer to the challenge of `06b_mcp_at_the_desk.ipynb`. The setup repeats what the notebook needs, so
this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import json
import os
import sys
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


## The server copy with `get_recent_invoices`

The copy is `solved/chinook_server_plus.py`: `chinook_server.py` with one more `TypedDict` and one more tool. We
print the two additions.

In [2]:
SERVER_FILE = ROOT / "06_MCP" / "solved" / "chinook_server_plus.py"

source = SERVER_FILE.read_text()
print(source[source.index("class RecentInvoice"):source.index("@mcp.tool()\ndef find_customer")].rstrip())
print()
print(source[source.index("@mcp.tool()\ndef get_recent_invoices"):source.index("if __name__")].rstrip())

class RecentInvoice(TypedDict):
    invoice_id: int
    date: str
    total: float

@mcp.tool()
def get_recent_invoices(customer_id: int, limit: int = 3) -> list[RecentInvoice]:
    """List a customer's most recent invoices, newest first, with their date and total in US dollars."""
    rows = con.execute(
        "SELECT InvoiceId, date(InvoiceDate), Total FROM invoices WHERE CustomerId = ? "
        "ORDER BY InvoiceDate DESC LIMIT ?",
        (customer_id, limit),
    ).fetchall()
    return [{"invoice_id": r[0], "date": r[1], "total": r[2]} for r in rows]


- `RecentInvoice` describes one record of the result: an `invoice_id`, a `date` and a `total`.
- `limit: int = 3` has a default value, so a caller may leave it out.
- The query orders the customer's invoices by date, newest first, and keeps the first `limit` rows.
  `date(InvoiceDate)` cuts the stored date and time down to `YYYY-MM-DD`.

## 1. List the copy's tools and call the new one by hand

In [3]:
from mcp import Client, StdioServerParameters

SERVER = StdioServerParameters(command=sys.executable, args=[str(SERVER_FILE)])

async with Client(SERVER) as mcp_client:
    listed = await mcp_client.list_tools()

print([tool.name for tool in listed.tools])
print(json.dumps(listed.tools[3].input_schema, indent=2))

['find_customer', 'get_invoices', 'get_support_rep', 'get_recent_invoices']
{
  "type": "object",
  "properties": {
    "customer_id": {
      "title": "Customer Id",
      "type": "integer"
    },
    "limit": {
      "default": 3,
      "title": "Limit",
      "type": "integer"
    }
  },
  "required": [
    "customer_id"
  ],
  "title": "get_recent_invoicesArguments"
}


Four tools. In the schema of `get_recent_invoices`, `limit` carries `"default": 3` and is missing from `required`,
which lists only `customer_id`. The server read both from the function's first line.

Now the call, for customer 6, without a `limit`.

In [4]:
async with Client(SERVER) as mcp_client:
    result = await mcp_client.call_tool("get_recent_invoices", {"customer_id": 6})

print("content blocks:", len(result.content))
for block in result.content:
    print("   ", block.text.replace("\n", " "))
print("structuredContent:", result.structured_content)

content blocks: 3
    {   "invoice_id": 404,   "date": "2013-11-13",   "total": 25.86 }
    {   "invoice_id": 393,   "date": "2013-10-03",   "total": 1.98 }
    {   "invoice_id": 272,   "date": "2012-04-11",   "total": 0.99 }
structuredContent: {'result': [{'invoice_id': 404, 'date': '2013-11-13', 'total': 25.86}, {'invoice_id': 393, 'date': '2013-10-03', 'total': 1.98}, {'invoice_id': 272, 'date': '2012-04-11', 'total': 0.99}]}


Three content blocks, one per invoice, newest first: invoice 404 (2013-11-13, 25.86), invoice 393 (2013-10-03,
1.98) and invoice 272 (2012-04-11, 0.99). `structuredContent` holds the same three records as a list under
`result`. The default did its work: we sent no `limit` and got three.

## 2. Give the copy to the Agents SDK

The host's code does what the notebook's does, with one change: the path in `MCPServerStdio` points at the copy. (The
model settings are written in the same cell here, where the notebook gives them a cell of their own.)

In [5]:
from agents import Agent, ModelSettings, Runner, set_default_openai_client
from agents.mcp import MCPServerStdio
from openai import AsyncOpenAI
from openai.types.shared import Reasoning

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))

INSTRUCTIONS = (
    "You help the support reps of a music shop. Look up every fact with the tools; "
    "never answer from memory. Answer in one or two sentences."
)

chinook_plus = MCPServerStdio(name="chinook", params={"command": sys.executable, "args": [str(SERVER_FILE)]})

desk = Agent(
    name="Desk",
    instructions=INSTRUCTIONS,
    model=MODEL,
    mcp_servers=[chinook_plus],
    model_settings=ModelSettings(reasoning=Reasoning(effort=EFFORT)),
)

async with chinook_plus:
    result = await Runner.run(desk, "What were Helena Holý's three most recent invoices?")

print(result.final_output)
print()
for item in result.new_items:
    if item.type == "tool_call_item":
        print("the model asked for:", item.raw_item.name, item.raw_item.arguments)

Helena Holý’s three most recent invoices were: Nov 13, 2013 — $25.86; Oct 3, 2013 — $1.98; and Apr 11, 2012 — $0.99.

the model asked for: find_customer {"name":"Helena Holý"}
the model asked for: get_recent_invoices {"customer_id":6,"limit":3}


The cell prints only the calls; the three invoices the tool sent back are the three blocks we read by hand in step 1.
Two calls: `find_customer` with Helena's name, then `get_recent_invoices` with `customer_id` 6. In our run the
model also wrote `"limit":3`, taking the "three" from the question. The answer gives the three dates and totals we
got by hand: 13 November 2013 (\$25.86), 3 October 2013 (\$1.98) and 11 April 2012 (\$0.99).

**Notice** what the answer left out in our run: the invoice numbers. The tool returned 404, 393 and 272, and the
model wrote a sentence about dates and amounts. A reply is the model's wording of a tool result. When a rep needs
the invoice number, say so in the instructions or read it from the tool's output.

## What changed, and what did not

One file changed: the server gained a `TypedDict` and a function. The host's code did not change, apart from the
path of the file it starts. The SDK asked the server for its tools at the start of the run, found four, and the
model used the new one. A new lookup is added in the server, and every host that uses the server gets it.